# Axe 3 : ce qui aide et ce qui gêne

**Question pour la DRH :** qu'est-ce qui facilite ou complique le travail au quotidien ?

Colonnes : `JobDifficulty` (ce qui a le plus compliqué le travail ces 6 derniers mois), `JobEase` (ce qui l'a le plus facilité), `JobSat` (satisfaction de 0 à 10). Questions nouvelles en 2026 : pas de comparaison avec 2025 (voir `HARMONISATION.md`).

In [ ]:
import sys
sys.path.append('..')  # pour trouver le dossier analyse/ depuis notebooks/

import pandas as pd
import plotly.express as px
from analyse.chargement import charger
from analyse.population import preparer

## 1. La population

La population commune vient de `preparer()` (socle) : elle ajoute `Zone`, `Tranche_experience` et `Secteur`.

Zone par défaut : France et pays frontaliers.

In [ ]:
df = preparer(charger(2026))

zone = df[df['Zone'].isin(['France', 'Pays frontaliers'])]
print('Population totale :', len(df))
print('France et pays frontaliers :', len(zone))
print('… qui ont répondu à JobDifficulty :', zone['JobDifficulty'].notna().sum())
print('… qui ont répondu à JobEase :', zone['JobEase'].notna().sum())

## 2. Les traductions pour l'affichage

Les réponses en anglais sont recopiées telles quelles depuis les données (attention à l'apostrophe typographique de `I’m not sure`).

In [ ]:
DIFFICULTES = {
    'Unclear priorities or requirements': 'Priorités floues',
    'Inefficient processes, workflows, or coordination': 'Organisation inefficace',
    'Not enough time, staffing, or resources': 'Manque de temps ou de moyens',
    'Leadership activity or organizational changes': 'Changements dans la direction',
    'Nothing has made my job harder in the past 6 months': "Rien n'a compliqué le travail",
    'Unreliable systems, infrastructure, or environments': 'Outils peu fiables',
    'Poor documentation or knowledge sharing': 'Documentation insuffisante',
    'Lack of support from teammates, managers, or mentors': 'Manque de soutien',
    'Lack of access to the right tools or platforms': 'Pas les bons outils',
    'I’m not sure': 'Ne sait pas',
    'Other, please specify': 'Autre',
}

AIDES = {
    'Access to the right tools or platforms': 'Les bons outils',
    'Supportive teammates, managers, or mentors': "Soutien de l'équipe",
    'Clear priorities or requirements': 'Priorités claires',
    'Better processes, workflows, or coordination': 'Meilleure organisation',
    'Enough time, staffing, or resources': 'Assez de temps ou de moyens',
    'Helpful documentation or knowledge sharing': 'Bonne documentation',
    'Reliable systems, infrastructure, or environments': 'Outils fiables',
    'Leadership activity or organizational changes': 'Changements dans la direction',
    'Nothing has made my job easier in the past 6 months': "Rien n'a facilité le travail",
    'I’m not sure': 'Ne sait pas',
    'Other, please specify': 'Autre',
}

# vérification : toutes les réponses des données sont-elles traduites ?
print(set(zone['JobDifficulty'].dropna()) - set(DIFFICULTES))
print(set(zone['JobEase'].dropna()) - set(AIDES))

## 3. Ce qui gêne : le classement

In [ ]:
def classement(donnees, colonne, traductions):
    reponses = donnees[colonne].dropna()
    tableau = reponses.value_counts().rename('repondants').to_frame()
    tableau['part'] = (tableau['repondants'] / len(reponses) * 100).round(1)
    tableau.index = tableau.index.map(traductions)
    return tableau

gene = classement(zone, 'JobDifficulty', DIFFICULTES)
gene

In [ ]:
px.bar(gene.iloc[::-1], x='part', orientation='h', text='part',
       title='Ce qui a le plus compliqué le travail (France et pays frontaliers)')

**Je remarque :** ce qui gêne le plus, c'est l'organisation du travail : les priorités floues et l'organisation inefficace (environ 17 % chacune), puis le manque de temps ou de moyens (14 %). Les outils gênent peu : seulement 2,5 % disent ne pas avoir les bons outils.

**Je me demande :** est-ce que les jeunes développeurs, ceux qui ont 2 ou 3 ans d'expérience, sont gênés par les mêmes choses ?

## 4. Ce qui aide : le classement

In [ ]:
aide = classement(zone, 'JobEase', AIDES)
aide

In [ ]:
px.bar(aide.iloc[::-1], x='part', orientation='h', text='part',
       title='Ce qui a le plus facilité le travail (France et pays frontaliers)')

**Je remarque :** ce qui aide le plus, ce sont les bons outils (19 %) et le soutien de l'équipe (17 %). Près d'une personne sur quatre (23 %) dit que rien n'a facilité son travail.

**Je me demande :** les personnes pour qui rien n'a facilité le travail sont-elles moins satisfaites que les autres ?

## 5. Satisfaction moyenne selon la difficulté principale

On compare la satisfaction moyenne de chaque groupe à celle de tous les répondants. Attention : c'est une association, pas une cause.

In [ ]:
avec_sat = zone.dropna(subset=['JobSat', 'JobDifficulty'])
moyenne_tous = avec_sat['JobSat'].mean()

sat = (avec_sat.groupby('JobDifficulty')['JobSat']
       .agg(satisfaction='mean', repondants='size')
       .sort_values('satisfaction'))
sat['satisfaction'] = sat['satisfaction'].round(2)
sat.index = sat.index.map(DIFFICULTES)
print('Satisfaction moyenne de tous :', round(moyenne_tous, 2))
sat

In [ ]:
fig = px.bar(sat, x='satisfaction', orientation='h', text='satisfaction',
             title='Satisfaction moyenne selon la difficulté principale')
fig.add_vline(x=moyenne_tous, line_color='gray', line_dash='dash')
fig

**Je remarque :** la satisfaction moyenne est de 7,0 sur 10. Ceux qui manquent surtout de soutien sont les moins satisfaits (6,0), suivis de ceux qui citent les changements dans la direction (6,3). Ceux qui manquent de temps restent au-dessus de la moyenne (7,3) : ce qui gêne souvent n'est pas forcément ce qui pèse le plus sur la satisfaction.

**Je me demande :** les groupes de moins de 50 répondants sont-ils fiables ? Ici, tous dépassent 50, mais ce ne sera plus le cas avec des filtres (France seule, ESN seule).

## 6. ESN et autres secteurs

La DRH dirige une ESN : la difficulté principale est-elle la même qu'ailleurs ?

In [ ]:
par_secteur = pd.concat({
    s: classement(zone[zone['Secteur'] == s], 'JobDifficulty', DIFFICULTES)['part']
    for s in ['ESN', 'Éditeur de logiciels', 'Autre secteur']
}, axis=1)
print('Répondants :', zone.dropna(subset=['JobDifficulty'])['Secteur'].value_counts().to_dict())
par_secteur

**Je remarque :** les ESN ont presque le même classement que les autres secteurs. Les priorités floues arrivent en tête (17 %). Le manque de soutien est un peu plus cité en ESN (4,3 %) que chez les éditeurs de logiciels (2,1 %), mais cela concerne peu de personnes.

**Je me demande :** en ESN, le développeur est souvent en mission chez un client : les priorités floues viennent-elles du client ou de l'ESN ? L'enquête ne permet pas de le savoir.

## 7. Ce que je retiens pour la page

- **Vue 1 :** le classement de ce qui gêne, avec un filtre pour passer à ce qui aide. Les 3 premières barres sont en couleur `ACCENT`.
- **Vue 2 :** la satisfaction moyenne selon la difficulté principale, avec une ligne grise pour la moyenne de tous.
- **Titre qui affirme :** « L'organisation des missions gêne plus que les outils ».
- **Message pour la DRH :** clarifier les priorités et l'organisation des missions, et soutenir les développeurs (managers, mentors), plutôt qu'investir d'abord dans les outils. Ce sont des liens observés chez des volontaires, pas des causes.

## 8. Test des fonctions de `analyse/axe_3.py`

Les fonctions de la page sont dans `analyse/axe_3.py`. On vérifie ici qu'elles donnent les mêmes chiffres que l'exploration ci-dessus.

In [ ]:
from analyse.axe_3 import (classement_difficultes, classement_aides,
                            satisfaction_par_difficulte, satisfaction_moyenne)

classement_difficultes(zone)

In [ ]:
classement_aides(zone)

In [ ]:
print('Satisfaction moyenne de tous :', satisfaction_moyenne(zone))
satisfaction_par_difficulte(zone)

In [ ]:
# mêmes chiffres que l'exploration ? (sections 3, 4 et 5)
assert classement_difficultes(zone)['repondants'].tolist() == gene['repondants'].tolist()
assert classement_aides(zone)['repondants'].tolist() == aide['repondants'].tolist()
assert satisfaction_par_difficulte(zone)['satisfaction'].tolist() == sat['satisfaction'].tolist()
print('Les fonctions donnent les mêmes chiffres que l\'exploration.')

In [ ]:
# avec un filtre plus fin, certains groupes passent sous le seuil de 50 répondants
from analyse.affichage import SEUIL

esn_france = df[(df['Zone'] == 'France') & (df['Secteur'] == 'ESN')]
resultat = satisfaction_par_difficulte(esn_france)
resultat['assez'] = resultat['repondants'] >= SEUIL
resultat

**Je remarque :** en France seule et en ESN, aucune difficulté n'atteint 50 répondants. Il faudra afficher « Effectif trop faible » dans la page avec ce filtre.